SỬ DỤNG PYTHON 3.12 TRÊN KAGGLE ĐỂ HUẤN LUYỆN

In [1]:
# =====================================================================
# CELL 1: CẬP NHẬT VÀ CÀI ĐẶT HỆ SINH THÁI HUGGING FACE (Lặng lẽ)
# =====================================================================
# Dùng tham số -q (quiet) để ẩn đi các dòng log cài đặt dài dòng
!pip install -q transformers datasets evaluate accelerate scikit-learn
print("✅ Cài đặt xong! Hệ sinh thái Hugging Face đã sẵn sàng.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
✅ Cài đặt xong! Hệ sinh thái Hugging Face đã sẵn sàng.


In [2]:
# =====================================================================
# CELL 2: NẠP THƯ VIỆN & KIỂM TRA PHẦN CỨNG (ZERO TRUST)
# =====================================================================
import os
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

# Chốt chặn kiểm tra: Ép hệ thống báo cáo đang dùng GPU hay CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Hệ thống tính toán đang chạy trên: [{device.type.upper()}]")

if device.type != 'cuda':
    print("❌ CẢNH BÁO: Chưa bật GPU! Hãy vào menu bên phải -> Accelerator -> Chọn GPU T4 x2.")
else:
    print(f"✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}")

🚀 Hệ thống tính toán đang chạy trên: [CUDA]
✅ GPU sẵn sàng: Tesla T4


In [3]:
# =====================================================================
# CELL 3: TỰ ĐỘNG TÌM FILE & NẠP TOKENIZER
# =====================================================================
import os
import pandas as pd
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

# 1. Zero Trust: Không gõ tay đường dẫn, bắt máy tự tìm file CSV
print("🔍 Đang rà quét toàn bộ kho dữ liệu Kaggle...")
train_path = None
dev_path = None

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'train_clean_PhoBERT.csv':
            train_path = os.path.join(dirname, filename)
        elif filename == 'dev_clean_PhoBERT.csv':
            dev_path = os.path.join(dirname, filename)

if not train_path or not dev_path:
    raise FileNotFoundError("❌ LỖI NGHIÊM TRỌNG: Không tìm thấy file. Bạn đã nhấn 'Add Input' để nạp dataset vào Notebook chưa?")

print(f"✅ Đã chốt tọa độ Train: {train_path}")
print(f"✅ Đã chốt tọa độ Dev: {dev_path}")

# 2. Cấu hình Mục tiêu học
TARGET_LABEL = "sentiment" # Nếu bạn muốn đoán chủ đề, đổi thành "topic"

print(f"\n📥 Đang nạp dữ liệu từ tọa độ đã chốt...")
df_train = pd.read_csv(train_path)
df_dev = pd.read_csv(dev_path)

# 3. Tự động mã hóa Nhãn (String -> Integer)
unique_labels = df_train[TARGET_LABEL].dropna().unique().tolist()
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"✅ Hệ thống nhận diện {len(unique_labels)} nhãn: {label2id}")

# Ép kiểu dữ liệu
df_train['labels'] = df_train[TARGET_LABEL].map(label2id)
df_dev['labels'] = df_dev[TARGET_LABEL].map(label2id)

# 4. Chuyển đổi sang Dataset Hugging Face
hg_dataset = DatasetDict({
    'train': Dataset.from_pandas(df_train[['clean_text_PhoBERT', 'labels']]),
    'validation': Dataset.from_pandas(df_dev[['clean_text_PhoBERT', 'labels']])
})

# 5. Kích hoạt Tokenizer
print("\n⏳ Đang tải PhoBERT Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

def tokenize_function(examples):
    return tokenizer(examples["clean_text_PhoBERT"], padding="max_length", truncation=True, max_length=256)

print("⚙️ Đang mã hóa hàng loạt văn bản (Batch Tokenization)...")
tokenized_datasets = hg_dataset.map(tokenize_function, batched=True)

print("🎉 Hoàn tất! Dữ liệu đã sẵn sàng để đưa vào GPU.")

🔍 Đang rà quét toàn bộ kho dữ liệu Kaggle...
✅ Đã chốt tọa độ Train: /kaggle/input/datasets/conbobietbay/phobert-student-feedback-clean/train_clean_PhoBERT.csv
✅ Đã chốt tọa độ Dev: /kaggle/input/datasets/conbobietbay/phobert-student-feedback-clean/dev_clean_PhoBERT.csv

📥 Đang nạp dữ liệu từ tọa độ đã chốt...
✅ Hệ thống nhận diện 3 nhãn: {2: 0, 0: 1, 1: 2}

⏳ Đang tải PhoBERT Tokenizer...


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

⚙️ Đang mã hóa hàng loạt văn bản (Batch Tokenization)...


Map:   0%|          | 0/11426 [00:00<?, ? examples/s]

Map:   0%|          | 0/1583 [00:00<?, ? examples/s]

🎉 Hoàn tất! Dữ liệu đã sẵn sàng để đưa vào GPU.


In [4]:
# =====================================================================
# CELL 4: KÍCH HOẠT HỆ THỐNG HUẤN LUYỆN (TRAINING CORE)
# =====================================================================
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

print(f"🧠 Đang nạp lõi PhoBERT với cấu hình {len(unique_labels)} nhãn phân loại...")
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/phobert-base-v2", 
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

# 3. Cấu hình Tham số
training_args = TrainingArguments(
    output_dir="./phobert_checkpoints",
    eval_strategy="epoch",          # <--- ĐÃ SỬA Ở ĐÂY (evaluation_strategy -> eval_strategy)
    save_strategy="epoch",          
    learning_rate=2e-5,             
    per_device_train_batch_size=32, 
    per_device_eval_batch_size=32,
    num_train_epochs=4,             
    weight_decay=0.01,
    fp16=True,                      
    load_best_model_at_end=True,    
    metric_for_best_model="f1_macro",
    report_to="none"                
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("\n🚀 BẮT ĐẦU HUẤN LUYỆN! HÃY THEO DÕI BẢNG THỐNG KÊ BÊN DƯỚI...\n")
trainer.train()

print("\n💾 Đang lưu mô hình tốt nhất ra thư mục Output...")
trainer.save_model("./best_phobert_model")
print("Hệ thống đã hoàn tất toàn bộ tiến trình.")

🧠 Đang nạp lõi PhoBERT với cấu hình 3 nhãn phân loại...


pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


💾 Đang lưu mô hình tốt nhất ra thư mục Output...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Hệ thống đã hoàn tất toàn bộ tiến trình.
